# COMP4328 Assignment 2: pilot and diagnostics

Select **Runtime > Change runtime type > L4 GPU**, then **Runtime > Run all**, and approve the Drive mount.

Expect roughly 1.5 to 2 hours, which uses Colab credits. Results save to Drive as they finish, so if the runtime stops, Run all again resumes from the last completed fit. The last cell prints ALL DONE.

Nothing here changes your earlier results. New files go to `results` in this notebook's `Tier 2-3 experiments` folder.

- **Part 1, pilot:** compares training settings on noisy validation only. It never evaluates clean test labels.
- **Part 2, diagnostics:** per-class accuracy from saved checkpoints, replay of the reported refinements with per-epoch logging, and a transition-estimator comparison.


In [1]:
import hashlib, json, os, re, shutil, subprocess, sys, zipfile
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import numpy, torch
DEVICE = os.environ.get("A2_DEVICE", "cuda")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime first: Runtime > Change runtime type > L4")
if "A2_DRIVE_DIR" not in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_DIR = Path(os.environ.get("A2_DRIVE_DIR", "/content/drive/MyDrive/Draft Assignment 2 COMP4328"))
WORK = DRIVE_DIR / "Tier 2-3 experiments"
if not WORK.is_dir():
    raise FileNotFoundError(f"Folder not found in this Google account: {WORK}")
ROOT = Path(os.environ.get("A2_ROOT", "/content/comp4328_tier23_v1"))
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT = WORK / "results"
OUTPUT.mkdir(exist_ok=True)
REFERENCE = DRIVE_DIR / "experiment_results_v2" / "final_v1"
GCE = DRIVE_DIR / "experiment_results_gce_v1" / "final_v1"
SMOKE = os.environ.get("A2_SMOKE", "") == "1"
def run(command):
    process = subprocess.Popen([str(c) for c in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=ROOT)
    for line in process.stdout:
        if not re.match(r"^\s+\w+\s+epoch \d+/", line):
            print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Failed with exit code {process.returncode}: {command[1]}")
def checked_copy(source, destination, expected):
    with Path(source).open("rb") as stream:
        if hashlib.file_digest(stream, "sha256").hexdigest() != expected:
            raise ValueError(f"Unexpected contents: {Path(source).name}")
    shutil.copyfile(source, destination)
environment = {"python": sys.version, "torch": torch.__version__, "numpy": numpy.__version__,
               "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None}
manifest = REFERENCE / "fashion0.6_given_trimmed" / "manifest.json"
reported = json.loads(manifest.read_text())["definition"]["environment"]
environment["matches_reported_runs"] = {k: environment[k] == reported[k] for k in ("python", "torch", "numpy", "gpu")}
(OUTPUT / "environment.json").write_text(json.dumps(environment, indent=1))
print(json.dumps(environment, indent=1))
if not all(environment["matches_reported_runs"].values()):
    print("NOTE: the environment differs from the reported runs, so exact replay is not expected. Continuing.")


Mounted at /content/drive
{
 "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
 "torch": "2.11.0+cu130",
 "numpy": "2.4.6",
 "gpu": "NVIDIA L4",
 "matches_reported_runs": {
  "python": true,
  "torch": true,
  "numpy": true,
  "gpu": true
 }
}


## Copy the code and datasets into runtime storage


In [2]:
CODE_SHA256 = "60d42f26a3d6d340053e114ec3b4cd26f0962a5b45d88654afd21c96ae305c25"
DATA_SHA256 = "e2ce73884b721359571c1f80f6e2d3510b1d21d57b8a1d6d1286f487657b7aff"
checked_copy(DRIVE_DIR / "assignment2_code_verified_v3.zip", ROOT / "source.zip", CODE_SHA256)
with zipfile.ZipFile(ROOT / "source.zip") as archive:
    archive.extractall(ROOT)
checked_copy(WORK / "pilot.py", ROOT / "pilot.py", "89f0f47dad847febfb28a6a9ff16a5f1fc5d26daa80b02efd11113a33f916f95")
checked_copy(WORK / "PLAN.md", ROOT / "PLAN.md", "8020d1cc6c219c7e7a123b1825e3ba673fa03ed6f39e305be432dd87cfd52931")
checked_copy(WORK / "diagnostics.py", ROOT / "algorithm" / "diagnostics.py", "adbbb4081846d9d4a50b6b2057635e47e9485cf651de75dda9129d771b1acddb")
checked_copy(DRIVE_DIR / "datasets.zip", ROOT / "datasets.zip", DATA_SHA256)
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
expected = {"FashionMNIST0.3.npz", "FashionMNIST0.6.npz", "CIFAR.npz"}
with zipfile.ZipFile(ROOT / "datasets.zip") as archive:
    for name in archive.namelist():
        if Path(name).name in expected:
            with archive.open(name) as source, (DATA / Path(name).name).open("wb") as target:
                shutil.copyfileobj(source, target)
if {p.name for p in DATA.glob("*.npz")} != expected:
    raise ValueError("Dataset archive is incomplete")
print("Code and datasets ready")


Code and datasets ready


## Part 1: blind pilot (about 20 to 30 minutes)

The rule in `PLAN.md` was fixed before any pilot result existed. Clean test labels are not evaluated here.


In [3]:
command = [sys.executable, "-u", ROOT / "pilot.py", "--data-dir", DATA, "--output", OUTPUT / "pilot", "--device", DEVICE]
if SMOKE:
    command += ["--splits", "0", "1", "--epochs", "2", "--sample-limit", "600"]
run(command)


cifar A_baseline split=0 loss=1.08959 noisy_acc=38.30 epoch=17 23s
cifar B_cosine split=0 loss=1.09018 noisy_acc=37.90 epoch=17 23s
cifar C_cosine_crop split=0 loss=1.09011 noisy_acc=38.50 epoch=17 26s
cifar D_cosine_crop_ema split=0 loss=1.09049 noisy_acc=37.70 epoch=10 26s
cifar E_resnet split=0 loss=1.08980 noisy_acc=38.10 epoch=23 89s
cifar A_baseline split=1 loss=1.09029 noisy_acc=38.37 epoch=13 23s
cifar B_cosine split=1 loss=1.09027 noisy_acc=38.07 epoch=13 23s
cifar C_cosine_crop split=1 loss=1.09068 noisy_acc=37.93 epoch=11 26s
cifar D_cosine_crop_ema split=1 loss=1.09061 noisy_acc=37.80 epoch=14 26s
cifar E_resnet split=1 loss=1.08989 noisy_acc=37.40 epoch=23 90s
cifar A_baseline split=2 loss=1.09192 noisy_acc=37.57 epoch=13 23s
cifar B_cosine split=2 loss=1.09041 noisy_acc=37.40 epoch=37 23s
cifar C_cosine_crop split=2 loss=1.09092 noisy_acc=37.47 epoch=37 26s
cifar D_cosine_crop_ema split=2 loss=1.09099 noisy_acc=37.47 epoch=40 26s
cifar E_resnet split=2 loss=1.08942 noisy_

## Part 2: diagnostics (about 1 to 1.5 hours)

These use the original model and settings. Selection stays on noisy validation inside the unchanged `fit()`; clean-test values are only logged.


In [4]:
command = [sys.executable, "-u", ROOT / "algorithm" / "diagnostics.py", "--task", "all", "--data-dir", DATA,
           "--output", OUTPUT / "diagnostics", "--device", DEVICE, "--reference-root", REFERENCE, "--gce-root", GCE]
if SMOKE:
    command += ["--splits", "0", "1", "--epochs", "2", "--sample-limit", "600", "--test-limit", "300"]
run(command)


per_class fashion0.3 {'ce': [98.4, 97.7, 94.1], 'forward': [98.4, 99.8, 98.8], 'reweight': [98.0, 99.5, 98.3], 'trevision': [98.4, 99.9, 98.3], 'gce': [99.1, 99.1, 95.5]}
per_class fashion0.6 {'ce': [86.8, 94.7, 84.4], 'forward': [90.1, 96.1, 89.1], 'reweight': [91.1, 95.4, 89.4], 'trevision': [94.9, 98.9, 93.4], 'gce': [89.7, 92.6, 87.7]}
per_class cifar {'ce': [56.2, 56.6, 67.0], 'forward': [53.0, 76.2, 79.5], 'reweight': [52.3, 72.3, 77.5], 'trevision': [52.3, 72.3, 77.5], 'gce': [53.0, 61.0, 65.6]}
fashion0.6_given_trimmed seed 0 start share below 0.594 reproduction {'reweight_difference': 0.0, 'ce_difference': 0.0, 'continued_difference': 0.0, 'continued_full_difference': 0.0, 'trevision_difference': 0.0, 'selected_epochs_match': True}
fashion0.6_given_trimmed seed 1 start share below 0.574 reproduction {'reweight_difference': 0.0, 'ce_difference': 0.0, 'continued_difference': 0.0, 'continued_full_difference': 0.0, 'trevision_difference': 0.0, 'selected_epochs_match': True}
fashio

## Finished check


In [5]:
needed = [OUTPUT / "environment.json", OUTPUT / "pilot" / "decision.json", OUTPUT / "diagnostics" / "per_class.json",
          OUTPUT / "diagnostics" / "gradient_summary.json", OUTPUT / "diagnostics" / "estimators_summary.json"]
missing = [p.name for p in needed if not p.is_file()]
if missing:
    raise RuntimeError(f"Missing outputs: {missing}")
summary = json.loads((OUTPUT / "diagnostics" / "gradient_summary.json").read_text())
for name, entry in summary.items():
    print(name, entry.get("reproduction", "no reproduction check"))
print("Pilot recommendation:", json.loads((OUTPUT / "pilot" / "decision.json").read_text())["recommended"])
print("ALL DONE. Results are in", OUTPUT)


fashion0.6_given_trimmed {'max_absolute_accuracy_difference': 0.0, 'selected_epochs_all_match': True}
cifar_estimated_trimmed {'max_absolute_accuracy_difference': 0.0, 'selected_epochs_all_match': True}
Pilot recommendation: None
ALL DONE. Results are in /content/drive/MyDrive/Draft Assignment 2 COMP4328/Tier 2-3 experiments/results
